In [1]:
import heapq
import math

# Hücre koordinatları (row, col)
def heuristic(a, b, kind="manhattan"):
    (x1, y1), (x2, y2) = a, b
    if kind == "euclidean":
        return math.hypot(x2 - x1, y2 - y1)
    # default: manhattan
    return abs(x1 - x2) + abs(y1 - y2)

def neighbors(node, grid, allow_diagonal=False):
    (r, c) = node
    rows, cols = len(grid), len(grid[0])
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if allow_diagonal:
        steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    result = []
    for dr, dc in steps:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
            result.append((nr, nc))
    return result

def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path

def astar(grid, start, goal, heuristic_kind="manhattan", allow_diagonal=False):
    """
    grid: 2D list where 0 = free cell, 1 = obstacle
    start, goal: (row, col)
    returns: path (list of nodes) or None if no path
    """
    open_set = []
    # heap elements: (f_score, g_score, node)
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal, heuristic_kind)}
    heapq.heappush(open_set, (f_score[start], g_score[start], start))

    came_from = {}

    closed_set = set()

    while open_set:
        _, current_g, current = heapq.heappop(open_set)

        if current == goal:
            return reconstruct_path(came_from, current)

        if current in closed_set:
            continue
        closed_set.add(current)

        for neighbor in neighbors(current, grid, allow_diagonal):
            tentative_g = g_score[current] + math.hypot(neighbor[0]-current[0], neighbor[1]-current[1])
            # if only 4-directional, distance is 1; with diagonal we used hypot (√2)
            if neighbor in g_score and tentative_g >= g_score[neighbor]:
                continue  # not a better path

            # this path is the best until now
            came_from[neighbor] = current
            g_score[neighbor] = tentative_g
            f = tentative_g + heuristic(neighbor, goal, heuristic_kind)
            f_score[neighbor] = f
            heapq.heappush(open_set, (f, tentative_g, neighbor))

    return None  # no path found

def print_grid_with_path(grid, path, start, goal):
    chars = {0: "·", 1: "█"}
    grid_vis = [[chars[cell] for cell in row] for row in grid]
    if path:
        for (r, c) in path:
            if (r, c) == start:
                grid_vis[r][c] = "S"
            elif (r, c) == goal:
                grid_vis[r][c] = "G"
            else:
                grid_vis[r][c] = "*"
    for row in grid_vis:
        print(" ".join(row))

if __name__ == "__main__":
    # Örnek ızgara: 0 = boş, 1 = engel
    example_grid = [
        [0,0,0,0,0,0,0,0],
        [0,1,1,1,0,1,1,1],
        [0,0,0,1,0,1,0,0],
        [0,1,0,0,0,1,0,0],
        [0,1,0,1,0,0,0,0],
        [0,0,0,1,0,1,1,0],
        [0,1,0,0,0,0,0,1],
        [0,0,0,0,1,0,0,0],
    ]
    start = (0, 0)
    goal = (7, 7)

    path = astar(example_grid, start, goal, heuristic_kind="manhattan", allow_diagonal=False)
    print("Bulunan yol (satır, sütun) biçiminde:", path)
    print()
    print_grid_with_path(example_grid, path, start, goal)

Bulunan yol (satır, sütun) biçiminde: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (6, 5), (6, 6), (7, 6), (7, 7)]

S * * * * · · ·
· █ █ █ * █ █ █
· · · █ * █ · ·
· █ · · * █ · ·
· █ · █ * · · ·
· · · █ * █ █ ·
· █ · · * * * █
· · · · █ · * G


In [2]:
import pygame
import math

# Ayarlar
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CAR_WIDTH = 40
CAR_HEIGHT = 80

class OtonomArac:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.angle = 0  # Aracın yönü (derece)
        self.velocity = 0
        self.steering_angle = 0 # Direksiyon açısı
        
        # Orijinal resim (Sprite)
        self.original_image = pygame.Surface((CAR_WIDTH, CAR_HEIGHT), pygame.SRCALPHA)
        pygame.draw.rect(self.original_image, (0, 0, 255), (0, 0, CAR_WIDTH, CAR_HEIGHT))
        self.image = self.original_image

    def update(self):
        # Basit Kinematik Model (Bicycle Model)
        # Hız ve yöne göre yeni konumu hesapla
        self.x += self.velocity * math.sin(math.radians(self.angle))
        self.y -= self.velocity * math.cos(math.radians(self.angle))
        
        # Direksiyon açısına göre aracın dönmesi
        # Gerçekte: açı += (hız / tekerlek_mesafesi) * tan(direksiyon_açısı)
        self.angle += self.steering_angle * self.velocity * 0.1 

    def draw(self, screen):
        # Aracı döndürerek çiz
        rotated_image = pygame.transform.rotate(self.original_image, -self.angle)
        rect = rotated_image.get_rect(center=(self.x, self.y))
        screen.blit(rotated_image, rect.topleft)
        
        # Sensörleri çiz (Lidar Simülasyonu)
        self.draw_sensors(screen)

    def draw_sensors(self, screen):
        # 5 adet Raycast (Işın) simülasyonu
        sensor_angles = [-30, -15, 0, 15, 30]
        sensor_length = 150
        
        for s_angle in sensor_angles:
            rad_angle = math.radians(self.angle + s_angle)
            end_x = self.x + math.sin(rad_angle) * sensor_length
            end_y = self.y - math.cos(rad_angle) * sensor_length
            pygame.draw.line(screen, (0, 255, 0), (self.x, self.y), (end_x, end_y), 1)

# Oyun Döngüsü
pygame.init()
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()
arac = OtonomArac(SCREEN_WIDTH/2, SCREEN_HEIGHT/2)

running = True
while running:
    screen.fill((50, 50, 50)) # Asfalt rengi
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Manuel Kontrol (Test için)
    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP]: arac.velocity += 0.1
    elif keys[pygame.K_DOWN]: arac.velocity -= 0.1
    else: arac.velocity *= 0.95 # Sürtünme
    
    if keys[pygame.K_LEFT]: arac.steering_angle = -5
    elif keys[pygame.K_RIGHT]: arac.steering_angle = 5
    else: arac.steering_angle = 0

    arac.update()
    arac.draw(screen)
    
    pygame.display.flip()
    clock.tick(60)

pygame.quit()

C:\Users\90534\AppData\Roaming\Python\Python312\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
import pygame
import sys

# --- Ayarlar ---
EKRAN_GENISLIK = 800
EKRAN_YUKSEKLIK = 600
FPS = 60
ARKA_PLAN_RENGI = (30, 30, 30) # Koyu gri
DIREKSIYON_RENGI = (200, 200, 200)
ISARET_RENGI = (255, 0, 0) # Dönüşü görmek için kırmızı şerit

# --- Pygame Başlatma ---
pygame.init()
screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
pygame.display.set_caption("2D Direksiyon Simülasyonu")
clock = pygame.time.Clock()

def direksiyon_olustur(cap):
    """
    Basit bir direksiyon yüzeyi (Surface) oluşturur.
    Resim yüklemek yerine çizim yaparak gösteriyoruz.
    """
    # Şeffaf bir yüzey oluştur
    surface = pygame.Surface((cap, cap), pygame.SRCALPHA)
    
    # 1. Dış Çember (Simit)
    pygame.draw.circle(surface, DIREKSIYON_RENGI, (cap//2, cap//2), cap//2, 20)
    
    # 2. İç Göbek
    pygame.draw.circle(surface, DIREKSIYON_RENGI, (cap//2, cap//2), 30)
    
    # 3. Kollar (Spokes) - Sol, Sağ ve Alt
    pygame.draw.rect(surface, DIREKSIYON_RENGI, (0, cap//2 - 10, cap, 20)) # Yatay kol
    pygame.draw.rect(surface, DIREKSIYON_RENGI, (cap//2 - 10, cap//2, 20, cap//2)) # Dikey alt kol

    # 4. Üstteki Kırmızı İşaret (Dönüşü anlamak için)
    pygame.draw.rect(surface, ISARET_RENGI, (cap//2 - 10, 0, 20, 30))
    
    return surface

# --- Hazırlık ---
# Direksiyonu bir kez oluşturuyoruz (veya pygame.image.load('resim.png') ile yükleyebilirsiniz)
orijinal_direksiyon = direksiyon_olustur(300)
direksiyon_rect = orijinal_direksiyon.get_rect(center=(EKRAN_GENISLIK//2, EKRAN_YUKSEKLIK//2))

aci = 0          # Mevcut açı
donus_hizi = 4   # Tuşa basınca ne kadar hızlı döneceği
toplanma_hizi = 2 # Tuşu bırakınca merkeze dönme hızı (Force Feedback simülasyonu gibi)
max_aci = 540    # Maksimum dönüş açısı (örn. 1.5 tur)

running = True
while running:
    # --- 1. Olayları Dinle ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # --- 2. Tuş Kontrolleri ---
    keys = pygame.key.get_pressed()
    
    # Sola Dönüş (Açıyı artır)
    if keys[pygame.K_LEFT] or keys[pygame.K_a]:
        if aci < max_aci:
            aci += donus_hizi
            
    # Sağa Dönüş (Açıyı azalt)
    elif keys[pygame.K_RIGHT] or keys[pygame.K_d]:
        if aci > -max_aci:
            aci -= donus_hizi
            
    # Tuşa basılmıyorsa direksiyonu yavaşça merkeze topla (Opsiyonel)
    else:
        if aci > 0:
            aci -= toplanma_hizi
            if aci < 0: aci = 0
        elif aci < 0:
            aci += toplanma_hizi
            if aci > 0: aci = 0

    # --- 3. Hesaplama ve Döndürme ---
    # Not: Pygame'de pozitif açı saatin tersi yönüdür (Counter-Clockwise)
    
    # Orijinal resmi döndürerek yeni bir yüzey oluşturuyoruz
    donmus_direksiyon = pygame.transform.rotate(orijinal_direksiyon, aci)
    
    # Yeni yüzeyin merkezini, eski yüzeyin merkezine eşitliyoruz.
    # BU ADIM ÇOK KRİTİKTİR. Yapmazsanız direksiyon ekranın sol üstüne doğru kayar.
    yeni_rect = donmus_direksiyon.get_rect(center=direksiyon_rect.center)

    # --- 4. Çizim ---
    screen.fill(ARKA_PLAN_RENGI)
    
    # Döndürülmüş resmi yeni koordinatına çiz
    screen.blit(donmus_direksiyon, yeni_rect)
    
    # Bilgi Yazısı (Debug)
    font = pygame.font.SysFont("Arial", 18)
    yazi = font.render(f"Açı: {int(aci)} derece", True, (255, 255, 255))
    screen.blit(yazi, (10, 10))

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()
sys.exit()

SystemExit: 

C:\Users\90534\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import pygame
import sys

# --- Ayarlar ---
EKRAN_GENISLIK = 800
EKRAN_YUKSEKLIK = 600
FPS = 60

# Renk Tanımları
RENK_CIM = (34, 139, 34)       # Arka plan
RENK_YOL = (100, 100, 100)     # Gri asfalt
RENK_PARK_ZEMIN = (120, 120, 120) # Park alanı biraz daha farklı gri
RENK_PARK_CIZGI = (220, 220, 220) # Park yeri çizgileri
RENK_ENGEL = (200, 50, 50)     # Kırmızı engeller
RENK_OYUNCU = (50, 100, 250)   # Mavi oyuncu karesi

# --- Pygame Başlatma ---
pygame.init()
screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
pygame.display.set_caption("2D Sabit Harita: Yollar, Park ve Engeller")
clock = pygame.time.Clock()

# --- HARİTA TANIMLARI (Dikdörtgen Listeleri) ---

# 1. Yollar (Asfalt alanlar)
# Rect(x, y, genişlik, yükseklik) formatında tanımlıyoruz.
yollar = [
    pygame.Rect(50, 50, 700, 80),   # Üst ana yol
    pygame.Rect(50, 470, 700, 80),  # Alt ana yol
    pygame.Rect(50, 130, 80, 340),  # Sol bağlantı yolu
    pygame.Rect(670, 130, 80, 340), # Sağ bağlantı yolu
    pygame.Rect(350, 250, 100, 220) # Ortadaki parka giden yol
]

# 2. Park Alanı
park_alani_ana = pygame.Rect(250, 150, 300, 150) # Parkın kendisi
park_yerleri_cizgileri = []
# Park alanının içine dikey çizgiler çizerek park yerlerini belirleyelim
for i in range(5):
    x_pos = 250 + (i + 1) * 50 # Her 50 pikselde bir çizgi
    # Çizgileri Rect olarak değil, çizim anında line olarak çizeceğiz
    park_yerleri_cizgileri.append(x_pos)


# 3. Engeller (İçinden geçilemeyecek alanlar)
engeller = [
    pygame.Rect(300, 70, 40, 40),   # Üst yolda bir kaza/kutu
    pygame.Rect(690, 300, 40, 80),  # Sağ yolda bir bariyer
    pygame.Rect(200, 200, 30, 30),  # Çimlerin üzerinde bir kaya
    pygame.Rect(360, 350, 80, 20)   # Park girişini daraltan bir engel
]

# --- OYUNCU TANIMI (Test için) ---
oyuncu = pygame.Rect(100, 80, 30, 30) # 30x30'luk bir kare
oyuncu_hizi = 4

def haritayi_ciz():
    """Tanımladığımız listeleri kullanarak haritayı ekrana çizer."""
    screen.fill(RENK_CIM) # En alta çim zemini döşe

    # Yolları çiz
    for yol in yollar:
        pygame.draw.rect(screen, RENK_YOL, yol)

    # Park alanını çiz
    pygame.draw.rect(screen, RENK_PARK_ZEMIN, park_alani_ana)
    # Park çizgilerini çiz (Sadece görsel)
    for x_pos in park_yerleri_cizgileri:
        # Park alanının üstünden altına çizgiler
        start_pos = (x_pos, park_alani_ana.top + 5)
        end_pos = (x_pos, park_alani_ana.bottom - 5)
        pygame.draw.line(screen, RENK_PARK_CIZGI, start_pos, end_pos, 3)

    # Engelleri çiz
    for engel in engeller:
        pygame.draw.rect(screen, RENK_ENGEL, engel)


# --- Ana Döngü ---
running = True
while running:
    # 1. Olay Kontrolü
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 2. Oyuncu Hareketi ve Çarpışma Kontrolü
    keys = pygame.key.get_pressed()
    hareket_x = 0
    hareket_y = 0

    if keys[pygame.K_LEFT]:  hareket_x = -oyuncu_hizi
    if keys[pygame.K_RIGHT]: hareket_x = oyuncu_hizi
    if keys[pygame.K_UP]:    hareket_y = -oyuncu_hizi
    if keys[pygame.K_DOWN]:  hareket_y = oyuncu_hizi

    # --- Kritik Kısım: Çarpışma Öncesi Kontrol ---
    # Oyuncuyu hemen hareket ettirmiyoruz. Önce "hareket ederse nerede olacak"
    # diye sanal bir dikdörtgen oluşturuyoruz.
    gelecek_konum_x = oyuncu.move(hareket_x, 0)
    gelecek_konum_y = oyuncu.move(0, hareket_y)

    # X ekseninde çarpışma var mı?
    carpisma_x = False
    for engel in engeller:
        if gelecek_konum_x.colliderect(engel):
            carpisma_x = True
            break # Bir engele çarptıysak diğerlerine bakmaya gerek yok
    
    # Y ekseninde çarpışma var mı?
    carpisma_y = False
    for engel in engeller:
        if gelecek_konum_y.colliderect(engel):
            carpisma_y = True
            break

    # Eğer çarpışma YOKSA, oyuncunun gerçek konumunu güncelle
    if not carpisma_x:
        oyuncu.x += hareket_x
    if not carpisma_y:
        oyuncu.y += hareket_y

    # Ekran dışına çıkmayı engelle
    oyuncu.clamp_ip(screen.get_rect())


    # 3. Çizim İşlemleri
    haritayi_ciz() # Önce harita
    pygame.draw.rect(screen, RENK_OYUNCU, oyuncu) # Sonra oyuncu (haritanın üstünde)

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()
sys.exit()

SystemExit: 

In [1]:
import pygame
import math

# --- Ayarlar ---
EKRAN_GENISLIK = 800
EKRAN_YUKSEKLIK = 600
FPS = 60

# Renkler
SIYAH = (0, 0, 0)
BEYAZ = (255, 255, 255)
KIRMIZI = (255, 0, 0)       # Lazer ışınları
YESIL = (0, 255, 0)         # Çarpışma noktaları (Point Cloud)
MAVI = (0, 0, 255)          # Araba
GRI = (100, 100, 100)       # Duvarlar

class LidarSensor:
    def __init__(self, menzil=200, cozunurluk=60):
        """
        menzil: Lidarın görebileceği maksimum piksel mesafesi
        cozunurluk: 360 derece içinde kaç tane ışın atılacağı (örn: 60 ışın)
        """
        self.menzil = menzil
        self.cozunurluk = cozunurluk
        self.veri = [] # (Açı, Mesafe, Koordinat) tutacak

    def tara(self, merkez_x, merkez_y, engeller):
        self.veri = [] # Her taramada veriyi sıfırla
        
        # 0'dan 360'a kadar belirtilen çözünürlükte döngü
        adim = 360 / self.cozunurluk
        
        for i in range(self.cozunurluk):
            aci_derece = i * adim
            aci_radyan = math.radians(aci_derece)
            
            # 1. Işının maksimum gideceği noktayı hesapla
            hedef_x = merkez_x + math.cos(aci_radyan) * self.menzil
            hedef_y = merkez_y + math.sin(aci_radyan) * self.menzil
            
            # En yakın çarpışmayı bulmak için başlangıçta max mesafeyi alıyoruz
            en_yakin_mesafe = self.menzil
            en_yakin_nokta = (hedef_x, hedef_y)
            carpisma_var = False

            # 2. Raycasting (Işın gönderme) - Engel Kontrolü
            # Pygame'in clipline fonksiyonu, bir çizginin dikdörtgen içinde kalan kısmını verir.
            # Bu, matematiksel kesişim formülleri yazmaktan çok daha hızlıdır.
            start_pos = (merkez_x, merkez_y)
            end_pos = (hedef_x, hedef_y)

            for engel in engeller:
                # clipline: Çizgi dikdörtgeni kesiyor mu?
                kesisim = engel.clipline(start_pos, end_pos)
                
                if kesisim:
                    # kesisim bir tuple döner: ((x1, y1), (x2, y2))
                    # Bize başlangıç noktasına (merkez_x, merkez_y) en yakın olan nokta lazım.
                    p1, p2 = kesisim
                    
                    # Mesafe hesapla (Pisagor)
                    mesafe1 = math.hypot(p1[0] - merkez_x, p1[1] - merkez_y)
                    mesafe2 = math.hypot(p2[0] - merkez_x, p2[1] - merkez_y)
                    
                    # Hangisi daha yakınsa onu al
                    if mesafe1 < mesafe2:
                        mevcut_mesafe = mesafe1
                        mevcut_nokta = p1
                    else:
                        mevcut_mesafe = mesafe2
                        mevcut_nokta = p2
                    
                    # Eğer bu engel, şu ana kadar bulduğumuz en yakın engelse kaydet
                    if mevcut_mesafe < en_yakin_mesafe:
                        en_yakin_mesafe = mevcut_mesafe
                        en_yakin_nokta = mevcut_nokta
                        carpisma_var = True

            # Veriyi kaydet: (Açı, Mesafe, Vurduğu Nokta X, Vurduğu Nokta Y)
            self.veri.append((aci_derece, en_yakin_mesafe, en_yakin_nokta, carpisma_var))

    def ciz(self, screen, merkez_x, merkez_y):
        # Işınları ve noktaları çiz
        for aci, mesafe, nokta, carpisma in self.veri:
            if carpisma:
                # Işın çizgisi (Faint Red)
                pygame.draw.line(screen, (50, 0, 0), (merkez_x, merkez_y), nokta, 1)
                # Vurduğu nokta (Point Cloud - Parlak Yeşil)
                pygame.draw.circle(screen, YESIL, (int(nokta[0]), int(nokta[1])), 3)
            else:
                # Çarpışma yoksa max menzile kadar silik çizgi
                pass 
                # pygame.draw.line(screen, (20, 20, 20), (merkez_x, merkez_y), nokta, 1)

# --- Ana Program ---
pygame.init()
screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
pygame.display.set_caption("LiDAR Simülasyonu")
clock = pygame.time.Clock()

# Harita (Engeller - Rect Listesi)
engeller = [
    pygame.Rect(0, 0, EKRAN_GENISLIK, 20), # Üst Duvar
    pygame.Rect(0, EKRAN_YUKSEKLIK-20, EKRAN_GENISLIK, 20), # Alt Duvar
    pygame.Rect(0, 0, 20, EKRAN_YUKSEKLIK), # Sol Duvar
    pygame.Rect(EKRAN_GENISLIK-20, 0, 20, EKRAN_YUKSEKLIK), # Sağ Duvar
    pygame.Rect(300, 200, 100, 100), # Orta Kutu
    pygame.Rect(600, 400, 50, 150),  # Sağ Engel
    pygame.Rect(150, 450, 200, 30)   # Alt Engel
]

# Araba ve Sensör
araba = pygame.Rect(100, 100, 40, 40)
# Menzil: 250px, Çözünürlük: 90 (Her 4 derecede bir ışın)
lidar = LidarSensor(menzil=250, cozunurluk=90) 

font = pygame.font.SysFont("Consolas", 14)

running = True
while running:
    # 1. Olaylar
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 2. Hareket (Mouse ile kontrol edelim ki test etmesi kolay olsun)
    mx, my = pygame.mouse.get_pos()
    araba.center = (mx, my)
    
    # Klavye ile kontrol etmek isterseniz burayı açın:
    # keys = pygame.key.get_pressed()
    # if keys[pygame.K_LEFT]: araba.x -= 5
    # ...

    # 3. LİDAR Taraması (Simülasyonun Kalbi)
    lidar.tara(araba.centerx, araba.centery, engeller)

    # 4. Çizim
    screen.fill(SIYAH)
    
    # Engelleri çiz
    for engel in engeller:
        pygame.draw.rect(screen, GRI, engel)
    
    # Arabayı çiz
    pygame.draw.rect(screen, MAVI, araba)
    
    # Lidarı çiz
    lidar.ciz(screen, araba.centerx, araba.centery)

    # 5. Veri Gösterimi (Ekrana Yazdırma)
    # Örnek olarak tam önündeki (0 derece) ve sağındaki (90 derece) mesafeyi yazdıralım
    # Lidar verisi [0] -> 0 derece, [cozunurluk/4] -> 90 derece civarıdır.
    
    if len(lidar.veri) > 0:
        on_mesafe = lidar.veri[0][1] # 0. indeks 0 derece kabul edilirse
        sag_mesafe = lidar.veri[int(lidar.cozunurluk/4)][1]
        
        text1 = font.render(f"Ön Mesafe (0°): {on_mesafe:.1f} px", True, BEYAZ)
        text2 = font.render(f"Sağ Mesafe (90°): {sag_mesafe:.1f} px", True, BEYAZ)
        
        screen.blit(text1, (10, 10))
        screen.blit(text2, (10, 30))

        # Terminale de veri akıtalım (Gerçek sensör simülasyonu)
        # print(f"LIDAR DATA: {lidar.veri[0]}") 

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()

C:\Users\90534\AppData\Roaming\Python\Python312\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
import pygame
import math
import random

# --- Ayarlar ---
EKRAN_GENISLIK = 800
EKRAN_YUKSEKLIK = 600
FPS = 60

# Renkler
SIYAH = (0, 0, 0)
BEYAZ = (255, 255, 255)
MAVI = (0, 100, 255)     # Araba
SARI = (255, 255, 0)     # GPS Rota İzi
GRI_YOL = (50, 50, 50)

class GPSSensor:
    def __init__(self, map_width, map_height):
        # REFERANS NOKTASI (Örnek: İstanbul Taksim Meydanı)
        # 0,0 noktası (Sol Üst) bu koordinatlara denk gelecek.
        self.ref_lat = 41.0369
        self.ref_lon = 28.9850
        
        # ÖLÇEK (Scale)
        # Ekrandaki 1 pikselin gerçekte kaç dereceye denk geldiği.
        # Basitlik için: Harita 800x600 piksel ise ve burası 200 metrelik bir alansa:
        # 1 derece enlem yaklaşık 111 km (111,000 metre) dir.
        # Bu değerlerle oynayarak haritanın "zoom" seviyesini ayarlıyoruz.
        self.scale_lat = 0.00001 # Her pikselde enlem değişimi
        self.scale_lon = 0.000015 # Her pikselde boylam değişimi
        
        # Gürültü (Noise)
        # Gerçek GPS mükemmel değildir, 1-2 metre hata payı olur.
        # Bunu simüle etmek için gürültü ekleyebiliriz (0 yaparsanız mükemmel çalışır).
        self.gurultu_seviyesi = 0.000002 
        
        self.current_lat = 0
        self.current_lon = 0
        self.rota_gecmisi = [] # Gittiği noktaları çizmek için

    def guncelle(self, x, y):
        """
        Piksel koordinatlarını (x, y) alır, Coğrafi koordinata çevirir.
        """
        # Gürültü ekle (Simülasyon gerçekçiliği için)
        noise_lat = random.uniform(-self.gurultu_seviyesi, self.gurultu_seviyesi)
        noise_lon = random.uniform(-self.gurultu_seviyesi, self.gurultu_seviyesi)
        
        # Dönüşüm Formülü: Referans + (Piksel * Ölçek)
        # Y ekseni aşağı indikçe artar ama Enlem yukarı gittikçe artar. 
        # Bu yüzden Y eksenini çıkarıyoruz veya ters çeviriyoruz.
        self.current_lat = self.ref_lat - (y * self.scale_lat) + noise_lat
        self.current_lon = self.ref_lon + (x * self.scale_lon) + noise_lon
        
        # Sadece çizim için piksel kaydı (Gürültülü veriyi kaydediyoruz)
        self.rota_gecmisi.append((x, y))
        
        # Rota çok uzarsa eski verileri sil (Performans için)
        if len(self.rota_gecmisi) > 500:
            self.rota_gecmisi.pop(0)

    def veri_oku(self):
        """Gerçek GPS verisi formatında string döner."""
        return {
            "lat": self.current_lat,
            "lon": self.current_lon,
            "format": f"LAT: {self.current_lat:.6f}, LON: {self.current_lon:.6f}"
        }

    def rotayi_ciz(self, screen):
        """Arabanın geçtiği yerleri çizer."""
        if len(self.rota_gecmisi) > 1:
            pygame.draw.lines(screen, SARI, False, self.rota_gecmisi, 2)

# --- Ana Program ---
pygame.init()
screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
pygame.display.set_caption("GPS Simülasyonu")
clock = pygame.time.Clock()
font = pygame.font.SysFont("Consolas", 16)

# Modülleri Başlat
araba = pygame.Rect(EKRAN_GENISLIK//2, EKRAN_YUKSEKLIK//2, 40, 40)
gps = GPSSensor(EKRAN_GENISLIK, EKRAN_YUKSEKLIK)

# Araba Hız Kontrolü
hiz = 0
aci = 0

running = True
while running:
    # 1. Olaylar
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 2. Kontroller (Yön Tuşları ile Araba Sürme)
    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP]: hiz = 3
    elif keys[pygame.K_DOWN]: hiz = -2
    else: hiz = 0 # Gazı bırakınca dur
    
    if keys[pygame.K_LEFT]: aci += 3
    if keys[pygame.K_RIGHT]: aci -= 3

    # 3. Fizik (Basit Hareket)
    # Açıyı Radyana çevir
    rad = math.radians(aci)
    
    # Hız ve Açıya göre X ve Y değişimi
    dx = math.cos(rad) * hiz
    dy = -math.sin(rad) * hiz # Ekran koordinatında Y yukarı negatiftir
    
    araba.x += dx
    araba.y += dy
    
    # Ekran dışına çıkmayı engelle
    araba.clamp_ip(screen.get_rect())

    # --- 4. GPS GÜNCELLEME (Kritik Nokta) ---
    # Arabanın merkez koordinatlarını GPS'e gönderiyoruz
    gps.guncelle(araba.centerx, araba.centery)
    gps_verisi = gps.veri_oku()

    # 5. Çizim
    screen.fill(GRI_YOL)
    
    # Grid (Enlem Boylam çizgileri gibi hayali çizgiler)
    for i in range(0, EKRAN_GENISLIK, 100):
        pygame.draw.line(screen, (70, 70, 70), (i, 0), (i, EKRAN_YUKSEKLIK))
    for i in range(0, EKRAN_YUKSEKLIK, 100):
        pygame.draw.line(screen, (70, 70, 70), (0, i), (EKRAN_GENISLIK, i))

    # GPS İzi (Araba nereden geçti?)
    gps.rotayi_ciz(screen)

    # Araba (Basit kare yerine yönünü görelim diye dönen yüzey)
    araba_surf = pygame.Surface((40, 20), pygame.SRCALPHA)
    pygame.draw.rect(araba_surf, MAVI, (0, 0, 40, 20), border_radius=5)
    pygame.draw.rect(araba_surf, BEYAZ, (30, 2, 8, 16)) # Ön farlar belli olsun
    donmus_araba = pygame.transform.rotate(araba_surf, aci)
    new_rect = donmus_araba.get_rect(center=araba.center)
    screen.blit(donmus_araba, new_rect)

    # 6. Bilgi Ekranı (HUD)
    # GPS Verisini Ekrana Yazdır
    panel = pygame.Surface((350, 80))
    panel.set_alpha(200)
    panel.fill(SIYAH)
    screen.blit(panel, (10, 10))
    
    text_lat = font.render(f"LAT: {gps_verisi['lat']:.6f} N", True, BEYAZ)
    text_lon = font.render(f"LON: {gps_verisi['lon']:.6f} E", True, BEYAZ)
    text_coord = font.render(f"Piksel: ({int(araba.centerx)}, {int(araba.centery)})", True, (200, 200, 200))
    
    screen.blit(text_lat, (20, 20))
    screen.blit(text_lon, (20, 40))
    screen.blit(text_coord, (20, 60))

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()

In [3]:
import pygame
import math
import random

# --- GENEL AYARLAR ---
EKRAN_GENISLIK = 1000
EKRAN_YUKSEKLIK = 700
FPS = 60

# --- RENKLER ---
SIYAH = (0, 0, 0)
BEYAZ = (255, 255, 255)
GRI_ZEMIN = (40, 40, 40)
GRI_DUVAR = (100, 100, 100)
GRI_YOL = (70, 70, 70)
MAVI_ARABA = (50, 100, 255)
KIRMIZI_LIDAR = (200, 50, 50, 100) # Şeffaf kırmızı
YESIL_POINTCLOUD = (0, 255, 0)
SARI_UI = (255, 255, 0)

# ==========================================
# MODÜL 1: HARİTA YAPISI
# ==========================================
class GameMap:
    def __init__(self):
        # Yollar (Sadece görsel zemin)
        self.yollar = [
            pygame.Rect(50, 100, 900, 150), # Üst ana yol
            pygame.Rect(50, 450, 900, 150), # Alt ana yol
            pygame.Rect(100, 100, 150, 500), # Sol bağlantı
            pygame.Rect(750, 100, 150, 500), # Sağ bağlantı
            pygame.Rect(400, 250, 200, 200)  # Orta meydan
        ]
        # Engeller (Çarpışılacak duvarlar)
        # Ekran sınırlarını da duvar olarak ekleyelim
        self.duvarlar = [
            pygame.Rect(0, 0, EKRAN_GENISLIK, 20), # Üst Sınır
            pygame.Rect(0, EKRAN_YUKSEKLIK-20, EKRAN_GENISLIK, 20), # Alt Sınır
            pygame.Rect(0, 0, 20, EKRAN_YUKSEKLIK), # Sol Sınır
            pygame.Rect(EKRAN_GENISLIK-20, 0, 20, EKRAN_YUKSEKLIK), # Sağ Sınır
            # İç engeller
            pygame.Rect(250, 250, 150, 50),
            pygame.Rect(600, 400, 50, 150),
            pygame.Rect(300, 50, 50, 50),
            pygame.Rect(450, 300, 100, 100) # Orta göbek
        ]

    def ciz(self, screen):
        screen.fill(GRI_ZEMIN)
        for yol in self.yollar:
            pygame.draw.rect(screen, GRI_YOL, yol)
        for duvar in self.duvarlar:
            pygame.draw.rect(screen, GRI_DUVAR, duvar)

# ==========================================
# MODÜL 2: LIDAR SENSÖRÜ
# ==========================================
class LidarSensor:
    def __init__(self, menzil=300, cozunurluk=120):
        self.menzil = menzil
        self.cozunurluk = cozunurluk # 360 derecede atılacak ışın sayısı
        self.veri = [] # (Açı, Mesafe, Nokta)

    def tara(self, merkez_x, merkez_y, araba_acisi, engeller):
        self.veri = []
        adim = 360 / self.cozunurluk
        
        # Arabanın baktığı yönü 0 derece kabul etmek için ofset ekliyoruz
        baslangic_acisi = araba_acisi - 90

        for i in range(self.cozunurluk):
            # Işın açısı = Arabanın açısı + tarama açısı
            aci_derece = baslangic_acisi + (i * adim)
            aci_radyan = math.radians(aci_derece)
            
            hedef_x = merkez_x + math.cos(aci_radyan) * self.menzil
            hedef_y = merkez_y - math.sin(aci_radyan) * self.menzil # Y ekseni ters
            
            en_yakin_mesafe = self.menzil
            en_yakin_nokta = (hedef_x, hedef_y)
            carpisma_var = False

            start_pos = (merkez_x, merkez_y)
            end_pos = (hedef_x, hedef_y)

            for engel in engeller:
                kesisim = engel.clipline(start_pos, end_pos)
                if kesisim:
                    p1, p2 = kesisim
                    # Başlangıca en yakın noktayı bul (Pisagor)
                    mesafe1 = math.hypot(p1[0] - merkez_x, p1[1] - merkez_y)
                    mesafe2 = math.hypot(p2[0] - merkez_x, p2[1] - merkez_y)
                    
                    if mesafe1 < mesafe2:
                        mevcut_mesafe, mevcut_nokta = mesafe1, p1
                    else:
                        mevcut_mesafe, mevcut_nokta = mesafe2, p2
                    
                    if mevcut_mesafe < en_yakin_mesafe:
                        en_yakin_mesafe = mevcut_mesafe
                        en_yakin_nokta = mevcut_nokta
                        carpisma_var = True

            # Veriyi kaydet (Bağıl açı, Mesafe, Koordinat, Çarpışma Durumu)
            rel_aci = (i * adim) # Arabaya göre bağıl açı
            self.veri.append((rel_aci, en_yakin_mesafe, en_yakin_nokta, carpisma_var))

    def ciz_debug(self, screen, merkez_x, merkez_y):
        # Işınları çizmek için ayrı bir şeffaf yüzey kullanalım
        lidar_surf = pygame.Surface((EKRAN_GENISLIK, EKRAN_YUKSEKLIK), pygame.SRCALPHA)
        
        for _, _, nokta, carpisma in self.veri:
            if carpisma:
                # Lazer Işını (Şeffaf kırmızı)
                pygame.draw.line(lidar_surf, KIRMIZI_LIDAR, (merkez_x, merkez_y), nokta, 1)
                # Point Cloud Noktası (Parlak Yeşil)
                pygame.draw.circle(screen, YESIL_POINTCLOUD, (int(nokta[0]), int(nokta[1])), 2)
        
        screen.blit(lidar_surf, (0,0))

# ==========================================
# MODÜL 3: GPS SENSÖRÜ
# ==========================================
class GPSSensor:
    def __init__(self):
        # Referans: İstanbul Beşiktaş
        self.ref_lat = 41.0422
        self.ref_lon = 29.0060
        # Ölçekleme (Harita boyutuna göre uydurma)
        self.scale = 0.00001
        self.gurultu = 0.000003 # 2-3 metre sapma simülasyonu
        self.current_lat = 0
        self.current_lon = 0

    def guncelle(self, x, y):
        noise_lat = random.uniform(-self.gurultu, self.gurultu)
        noise_lon = random.uniform(-self.gurultu, self.gurultu)
        
        # Y ekseni aşağı indikçe enlem azalır
        self.current_lat = self.ref_lat - (y * self.scale) + noise_lat
        self.current_lon = self.ref_lon + (x * self.scale) + noise_lon

# ==========================================
# ANA UYGULAMA (SIMULATOR)
# ==========================================
class Simulator:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
        pygame.display.set_caption("Tam Otonom Araç Simülasyonu v1.0")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("Consolas", 14)
        self.font_big = pygame.font.SysFont("Consolas", 18, bold=True)

        # Modülleri Başlat
        self.map = GameMap()
        self.lidar = LidarSensor()
        self.gps = GPSSensor()

        # Araba Ayarları
        self.car_pos = pygame.Vector2(150, 175) # Başlangıç pozisyonu
        self.car_vel = pygame.Vector2(0, 0)     # Hız vektörü
        self.car_angle = 0 # Derece cinsinden yön (0 = Yukarı)
        self.car_speed = 0
        self.max_speed = 5
        self.rotation_speed = 3
        self.friction = 0.1 # Sürtünme

        # Araba Görseli Oluştur
        self.car_width = 40
        self.car_height = 20
        self.original_car_surf = pygame.Surface((self.car_width, self.car_height), pygame.SRCALPHA)
        pygame.draw.rect(self.original_car_surf, MAVI_ARABA, (0, 0, self.car_width, self.car_height), border_radius=4)
        pygame.draw.rect(self.original_car_surf, SARI_UI, (32, 2, 6, 16)) # Ön farlar (yönü belli etsin)
        self.car_rect = self.original_car_surf.get_rect(center=self.car_pos)

    def handle_input(self):
        keys = pygame.key.get_pressed()
        # Gaz / Fren (Hızlanma mantığı)
        if keys[pygame.K_UP]:
            self.car_speed += 0.2
            if self.car_speed > self.max_speed: self.car_speed = self.max_speed
        elif keys[pygame.K_DOWN]:
            self.car_speed -= 0.2
            if self.car_speed < -self.max_speed/2: self.car_speed = -self.max_speed/2 # Geri vites daha yavaş
        else:
            # Sürtünme ile yavaşlama
            if self.car_speed > 0:
                self.car_speed -= self.friction
                if self.car_speed < 0: self.car_speed = 0
            elif self.car_speed < 0:
                self.car_speed += self.friction
                if self.car_speed > 0: self.car_speed = 0

        # Direksiyon (Sadece hareket ederken dönebilir)
        if abs(self.car_speed) > 0.1:
            if keys[pygame.K_LEFT]:
                self.car_angle += self.rotation_speed * (1 if self.car_speed > 0 else -1)
            if keys[pygame.K_RIGHT]:
                self.car_angle -= self.rotation_speed * (1 if self.car_speed > 0 else -1)

    def update(self):
        # 1. FİZİK VE HAREKET
        rad = math.radians(self.car_angle)
        # Yön vektörünü hesapla (Pygame'de 0 derece sağdır, biz yukarı kabul ettik o yüzden dönüştürüyoruz)
        dir_vec = pygame.Vector2(math.cos(rad), -math.sin(rad))
        self.car_vel = dir_vec * self.car_speed
        
        # Tahmini yeni pozisyon
        new_pos = self.car_pos + self.car_vel
        
        # 2. ÇARPIŞMA KONTROLÜ (Collision Detection)
        temp_rect = self.car_rect.copy()
        temp_rect.center = new_pos
        
        carpisma = False
        for duvar in self.map.duvarlar:
            if temp_rect.colliderect(duvar):
                carpisma = True
                # Basit tepki: Hızı sıfırla ve biraz geri tep
                self.car_speed = -self.car_speed * 0.3
                break
        
        if not carpisma:
            self.car_pos = new_pos
            self.car_rect.center = self.car_pos

        # 3. SENSÖRLERİ GÜNCELLE
        # Lidar'a arabanın merkezini, açısını ve haritadaki duvarları gönder
        self.lidar.tara(self.car_pos.x, self.car_pos.y, self.car_angle, self.map.duvarlar)
        # GPS'e arabanın merkezini gönder
        self.gps.guncelle(self.car_pos.x, self.car_pos.y)

    def draw_ui(self):
        # Sol Üst: GPS Paneli
        pygame.draw.rect(self.screen, (0,0,0,180), (10, 10, 250, 80))
        gps_title = self.font_big.render("GPS MODÜLÜ", True, SARI_UI)
        lat_txt = self.font.render(f"LAT: {self.gps.current_lat:.6f} N", True, BEYAZ)
        lon_txt = self.font.render(f"LON: {self.gps.current_lon:.6f} E", True, BEYAZ)
        self.screen.blit(gps_title, (20, 15))
        self.screen.blit(lat_txt, (20, 40))
        self.screen.blit(lon_txt, (20, 60))

        # Sağ Üst: LiDAR Veri Paneli
        if len(self.lidar.veri) > 0:
            pygame.draw.rect(self.screen, (0,0,0,180), (EKRAN_GENISLIK-260, 10, 250, 100))
            lidar_title = self.font_big.render("LIDAR VERİSİ", True, YESIL_POINTCLOUD)
            # Veri yapısı: (bağıl_açı, mesafe, nokta, çarpışma)
            # 0. indeks tam önü (0 derece), Çözünürlük/4 tam solu (90 derece) temsil eder.
            on_mesafe = self.lidar.veri[0][1] 
            sol_mesafe = self.lidar.veri[int(self.lidar.cozunurluk/4)][1]
            sag_mesafe = self.lidar.veri[int(self.lidar.cozunurluk*3/4)][1]

            on_txt = self.font.render(f"ÖN (0°): {on_mesafe:.1f} px", True, BEYAZ)
            sol_txt = self.font.render(f"SOL(90°): {sol_mesafe:.1f} px", True, BEYAZ)
            sag_txt = self.font.render(f"SAĞ(-90°): {sag_mesafe:.1f} px", True, BEYAZ)

            self.screen.blit(lidar_title, (EKRAN_GENISLIK-250, 15))
            self.screen.blit(on_txt, (EKRAN_GENISLIK-250, 40))
            self.screen.blit(sol_txt, (EKRAN_GENISLIK-250, 60))
            self.screen.blit(sag_txt, (EKRAN_GENISLIK-250, 80))

    def run(self):
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            self.handle_input()
            self.update()

            # --- ÇİZİM ---
            self.map.ciz(self.screen) # 1. Harita zemini
            
            # 2. Lidar ışınları (Arabanın altında kalsın diye önce çiziyoruz)
            self.lidar.ciz_debug(self.screen, self.car_pos.x, self.car_pos.y)

            # 3. Araba (Döndürerek çiz)
            rotated_car = pygame.transform.rotate(self.original_car_surf, self.car_angle)
            rotated_rect = rotated_car.get_rect(center=self.car_pos)
            self.screen.blit(rotated_car, rotated_rect)
            
            # 4. UI Panelleri
            self.draw_ui()

            pygame.display.flip()
            self.clock.tick(FPS)
        pygame.quit()

# Kodu çalıştır
if __name__ == "__main__":
    sim = Simulator()
    sim.run()